In [3]:
import pandas as pd
import itertools

def american_to_decimal(american_odds):
    """Convert American odds to decimal odds."""
    if american_odds > 0:
        return 1 + (american_odds / 100)
    else:
        return 1 + (100 / abs(american_odds))

def find_all_arbitrage(df):
    arbs = []

    # --- Normalize data ---
    df = df.copy()
    df["Outcome"] = df["Outcome"].astype(str).str.strip().str.lower()  # normalize outcome names
    df["Bookmaker"] = df["Bookmaker"].astype(str).str.strip()
    df["Odds"] = pd.to_numeric(df["Odds"], errors="coerce")
    df = df.dropna(subset=["Odds"])  # remove rows with invalid odds

    # Group by Event only, ignoring Market
    grouped = df.groupby("Event")

    for event, group in grouped:
        outcomes = group["Outcome"].unique()
        num_outcomes = len(outcomes)

        # Skip events with less than 2 outcomes
        if num_outcomes < 2:
            continue

        # Build list of all bookmaker odds per outcome
        odds_by_outcome = {}
        for outcome in outcomes:
            odds_by_outcome[outcome] = group[group["Outcome"] == outcome][
                ["Bookmaker", "Odds"]
            ].to_records(index=False)

        # Cartesian product of bookmakers across outcomes
        outcome_combos = list(itertools.product(*odds_by_outcome.values()))

        for combo in outcome_combos:
            decimal_odds = []
            implied_prob = 0
            arb_row = {
                "Event": event,
                "Number of Outcomes": num_outcomes,
                "Outcomes": [],
                "Bookmakers": [],
                "Odds": [],
                "Recommended Stakes": [],
            }

            for (book, odds), outcome in zip(combo, odds_by_outcome.keys()):
                dec = american_to_decimal(odds)
                decimal_odds.append(dec)
                implied_prob += 1 / dec
                arb_row["Outcomes"].append(outcome)
                arb_row["Bookmakers"].append(book)
                arb_row["Odds"].append(odds)

            # Only keep true arbitrages
            if implied_prob >= 1:
                continue

            # Calculate proportional stakes for any total stake (here unitless)
            stakes = [(1 / d) / implied_prob for d in decimal_odds]

            arb_row["Recommended Stakes"] = stakes
            arb_row["Total Probability"] = round(implied_prob, 6)
            arb_row["Profit Margin (%)"] = round((1 - implied_prob) * 100, 6)
            # Guaranteed profit using smallest possible payout
            arb_row["Guaranteed Profit"] = round(min([s * d for s, d in zip(stakes, decimal_odds)]) - sum(stakes), 6)

            arbs.append(arb_row)

    return arbs

def main():
    input_path = r"D:\Sports Betting Python\Arbitrage\Arbitrage Input.xlsx"
    output_path = r"D:\Sports Betting Python\Arbitrage\Arbitrage Output.xlsx"

    df = pd.read_excel(input_path)

    required_cols = ["Event", "Outcome", "Bookmaker", "Odds"]
    if not all(c in df.columns for c in required_cols):
        raise ValueError(f"Missing required columns: {required_cols}")

    arbs = find_all_arbitrage(df)

    if not arbs:
        print("No arbitrage opportunities found.")
        return

    arb_df = pd.DataFrame(arbs)

    # Make lists readable in Excel
    arb_df["Outcomes"] = arb_df["Outcomes"].apply(lambda x: ", ".join(x))
    arb_df["Bookmakers"] = arb_df["Bookmakers"].apply(lambda x: ", ".join(x))
    arb_df["Odds"] = arb_df["Odds"].apply(lambda x: ", ".join(map(str, x)))
    arb_df["Recommended Stakes"] = arb_df["Recommended Stakes"].apply(
        lambda x: ", ".join([f"{round(v, 4)}" for v in x])
    )

    arb_df.to_excel(output_path, index=False)
    print(f"Arbitrage opportunities saved to:\n{output_path}")

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Sports Betting Python\\Arbitrage\\Arbitrage Input.xlsx'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 17.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 32.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/usr/local/python/3.12.1/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
